### Imports

In [1]:
import subprocess
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from datetime import datetime
import json


from models.archs.ConvNet import ConvNet
from config import parameters
from unlearn.GA import GA
from data.utils import get_unlearn_loader, setup_model_dataset, accuracy, retain_to_train_and_val
from trainer.utils import get_memory_footprint, get_model_weight_norm, check_accuracy, training_regimen

### Further split retain set into train and val

### Data loaders

In [2]:
# from torchvision.utils import make_grid
# # # ... define a dataset, forget set, and retain set
# _, _, _, marked_loader = setup_model_dataset(args = parameters)
# forget_loader, retain_loader = get_unlearn_loader(marked_loader = marked_loader, args = parameters)
# train_loader, val_loader = retain_to_train_and_val(retain_loader, seed = 42)

# X, y = next(iter(train_loader))
# plt.figure(figsize=(18,10))
# plt.imshow(make_grid(X, nrow=10).permute((1,2,0)));

### Run it

In [ ]:
import random

def retrain_runs(seed = 42, num_runs_each = 2, num_epochs = 20, results_folder = "results", checkpoints_folder = "models/model_checkpoints"):

    # apply seed to params, so all downstream functions can use it
    parameters["seed"] = seed

    # Make results_folder if it doesnt already exist
    if not os.path.exists(results_folder):
        print(f"{results_folder} doesn't exist - creating it...")
        os.makedirs(results_folder, exist_ok=True)

    # Make checkpoints_folder if it doesnt already exist
    if not os.path.exists(checkpoints_folder):
        print(f"{checkpoints_folder} doesn't exist - creating it...")
        os.makedirs(checkpoints_folder, exist_ok=True)

    # Save the config for this run to the results folder
    retrain_config = {
        "num_runs_each": num_runs_each,
        "num_epochs": num_epochs,
        "starting_LR": 1e-3,
        "weight_decay": 7e-4,
        "lr_scheduler_params": {
            "factor": 0.5,
            "patience": 2,
            "threshold": .01,
        }
    }

    with open(os.path.join(results_folder, "retrain_config.json"), "w") as f:
        json.dump(retrain_config, f, indent=4)

    # For each class ...
    for c in range(0, 10):

        print("="*50 + "    " + f"CLASS {c}\n")

        #... assign it as class to forget
        parameters["class_to_replace"] = c
        print(f"Class to forget for this run: {c}\n")

        # ... and do some number of runs, where
        for i in range(1, num_runs_each+1):

            print("-"*25 + "    " + f"RUN {i}\n")
            

            # ... init an empty model
            device = torch.device("mps") if torch.mps.is_available() else torch.device("cpu")
            base_model = ConvNet()
            base_model.to(device)
        
            # ... define data loaders w.r.t. class to forget
            # ----- we are deliberately defining these inside the loop, so that they are refreshed for each run, hence more random
            _, _, _, marked_loader = setup_model_dataset(args = parameters)
            forget_loader, retain_loader = get_unlearn_loader(marked_loader = marked_loader, args = parameters)
            train_loader, val_loader = retain_to_train_and_val(retain_loader, seed =int(f"{seed}{i}{c}"))
            
            # ... define params
            opt = optim.AdamW(base_model.parameters(), lr=retrain_config["starting_LR"], weight_decay = retrain_config["weight_decay"])
            criterion = nn.CrossEntropyLoss()
            scheduler = ReduceLROnPlateau(
                opt, 
                mode='min', 
                factor=retrain_config["lr_scheduler_params"]["factor"], 
                patience=retrain_config["lr_scheduler_params"]["patience"], 
                threshold = retrain_config["lr_scheduler_params"]["threshold"], 
                threshold_mode = "rel"
                )
            
            run_name = f"class_{c}_retrain_{i}"
            model_path = os.path.join(checkpoints_folder, f"{run_name}.pth")

            # ... and run the training regimen
            _, _, _ = training_regimen(
                base_model, 
                train_loader, 
                val_loader, 
                opt, 
                criterion, 
                scheduler, 
                device, 
                num_epochs = num_epochs, 
                best_val_loss = torch.inf, 
                model_path = model_path
                )
            
            # ... pull the best performing checkpoint at `model_path`
            best_model = ConvNet()
            checkpoint = torch.load(model_path)
            best_model.load_state_dict(checkpoint['model_state_dict'])
            best_model.eval()
            best_model.to(device)

            # ... and evaluate metrics
            forget_acc = check_accuracy(base_model, data_loader = forget_loader, device = device)
            retain_acc = check_accuracy(base_model, data_loader = val_loader, device = device)
            test_acc = check_accuracy(base_model, data_loader = val_loader, device = device) # THIS WILL EVENTUALLY NEED TO BE TEST_LOADER, ONCE WE FIGURE OUT HOW TO DO THAT APPROPRIATELY

            # ... bind results to a dictionary,
            results = {
                    "name": run_name,
                    "class": c,
                    "run": i,
                    "forget_acc": forget_acc,
                    "retain_acc": retain_acc,
                    "test_acc": test_acc       
            }

            # ... and save them
            with open(os.path.join(results_folder, f"{run_name}.json"), "w") as f:
                json.dump(results, f, indent=4)


In [ ]:
# MAKE A RANDOM SEED
seed = 22
# DO EXP
retrain_runs(seed = seed, num_runs_each = 1, num_epochs = 30, results_folder = f"results/seed_{seed}/retrain", checkpoints_folder = f"models/model_checkpoints/seed_{seed}/retrain")

==================================================    CLASS 0

Class to forget for this run: 0

-------------------------    RUN 1

Dataset information: CIFAR-10	 45000 images for training 	 5000 images for validation	
10000 images for testing	 no normalize applied in data_transform
Training data augmentation = randomcrop(32,4) + randomhorizontalflip


/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Dataset information: CIFAR-10	 45000 images for training 	 5000 images for validation	
10000 images for testing	 no normalize applied in data_transform
Training data augmentation = randomcrop(32,4) + randomhorizontalflip
datasets length:  4500 40500
SPLITTING RETAIN SET INTO TRAIN AND VAL:

Dataset information: CIFAR-10	 36450 images for training 	 4050 images for validation	
 ----- EPOCH 0 ----- 

Batch 0: Loss = 2.5253
Batch 100: Loss = 1.4724
Batch 200: Loss = 1.4489
Epoch 0 | LR: 1.0e-03 | Val Loss: 1.3063, Val Acc: 0.53 | RAM: 0.44GB | VRAM: 1.23GB | Weight Norm: 40.225
--- Epoch 0: New best model saved! ---
 ----- EPOCH 1 ----- 

Batch 0: Loss = 1.4020
Batch 100: Loss = 1.0522
Batch 200: Loss = 1.2338
Epoch 1 | LR: 1.0e-03 | Val Loss: 0.9259, Val Acc: 0.66 | RAM: 0.47GB | VRAM: 1.23GB | Weight Norm: 43.338
--- Epoch 1: New best model saved! ---
 ----- EPOCH 2 ----- 

Batch 0: Loss = 0.9293
Batch 100: Loss = 0.9233
Batch 200: Loss = 0.7627
Epoch 2 | LR: 1.0e-03 | Val Loss: 0.8529,

/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


==================================================    CLASS 1

Class to forget for this run: 1

-------------------------    RUN 1

Dataset information: CIFAR-10	 45000 images for training 	 5000 images for validation	
10000 images for testing	 no normalize applied in data_transform
Training data augmentation = randomcrop(32,4) + randomhorizontalflip
Dataset information: CIFAR-10	 45000 images for training 	 5000 images for validation	
10000 images for testing	 no normalize applied in data_transform
Training data augmentation = randomcrop(32,4) + randomhorizontalflip
datasets length:  4500 40500
SPLITTING RETAIN SET INTO TRAIN AND VAL:

Dataset information: CIFAR-10	 36450 images for training 	 4050 images for validation	
 ----- EPOCH 0 ----- 

Batch 0: Loss = 2.4840
Batch 100: Loss = 1.5520
Batch 200: Loss = 1.3452
Epoch 0 | LR: 1.0e-03 | Val Loss: 1.2366, Val Acc: 0.56 | RAM: 0.74GB | VRAM: 1.24GB | Weight Norm: 40.055
--- Epoch 0: New best model saved! ---
 ----- EPOCH 1 ----- 

Bat